# Day 10: End-to-End Orchestration, Observability & Verification

Complete E2E run of a synthetic patient from raw HL7/JSON -> Bronze -> Tokenization -> Silver OMOP -> DQ -> Gold readmission metric.

In [ ]:
import os
import json
import uuid
import hashlib
import duckdb
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

os.environ['SPARK_VERSION'] = '3.3'
os.environ['EHDIP_HASH_SALT'] = 'e2e_salt_999'

spark = SparkSession.builder \
    .appName("Day10_E2E_Pipeline") \
    .getOrCreate()

print("SparkSession initialized for E2E Pipeline Walkthrough.")

In [ ]:
# 1. Raw Ingestion -> Bronze
print("--- Step 1: Ingestion ---")
raw_json_1 = json.dumps({"patient_id": "P01", "mrn": "111-222", "diagnosis_code": "I10", "date": "2023-01-01"})
raw_json_2 = json.dumps({"patient_id": "P01", "mrn": "111-222", "diagnosis_code": "I10", "date": "2023-01-15"}) # 14 day readmit

df_bronze = spark.createDataFrame([
    (str(uuid.uuid4()), "EHR_SYSTEM", raw_json_1),
    (str(uuid.uuid4()), "EHR_SYSTEM", raw_json_2)
], ["payload_id", "source", "raw_json"])

df_bronze.show(truncate=False)

In [ ]:
# 2. Parse & De-Identify
print("--- Step 2: De-Identification ---")
df_parsed = df_bronze.withColumn("patient_id", F.get_json_object("raw_json", "$.patient_id")) \
                     .withColumn("mrn", F.get_json_object("raw_json", "$.mrn")) \
                     .withColumn("diagnosis_code", F.get_json_object("raw_json", "$.diagnosis_code")) \
                     .withColumn("date", F.to_date(F.get_json_object("raw_json", "$.date")))

def fpe_mask(val): return hashlib.sha256((val + os.environ['EHDIP_HASH_SALT']).encode()).hexdigest()[:10] if val else None
fpe_udf = F.udf(fpe_mask, StringType())

df_deid = df_parsed.withColumn("mrn_masked", fpe_udf("mrn"))
df_deid.select("patient_id", "mrn_masked", "diagnosis_code", "date").show()

In [ ]:
# 3. OMOP Silver Transformation & DQ
print("--- Step 3: Silver Transformation (OMOP) & DQ ---")
# Map I10 to OMOP concept 32020
df_silver = df_deid.withColumn("condition_concept_id", F.when(F.col("diagnosis_code") == "I10", 32020).otherwise(0))
df_silver = df_silver.select("patient_id", "condition_concept_id", F.col("date").alias("condition_start_date"))

# DQ Check (Ensure valid concept and date)
df_silver_valid = df_silver.filter((F.col("condition_concept_id") > 0) & (F.col("condition_start_date") <= F.current_date()))
df_silver_valid.show()

In [ ]:
# 4. Gold Metric Calculation (DuckDB Simulation for Snowflake/dbt)
print("--- Step 4: Gold Marts (Readmission Metric) ---")
con = duckdb.connect(':memory:')
silver_df = df_silver_valid.toPandas()
con.register('silver_table', silver_df)

gold_query = """
    WITH index_admissions AS (
        SELECT patient_id, condition_start_date AS index_date
        FROM silver_table WHERE condition_concept_id = 32020
    ),
    subsequent_admissions AS (
        SELECT patient_id, condition_start_date AS readmission_date
        FROM silver_table WHERE condition_concept_id = 32020
    )
    SELECT 
        i.patient_id,
        i.index_date,
        s.readmission_date,
        CAST(s.readmission_date - CAST(i.index_date AS DATE) AS INT) AS days_between
    FROM index_admissions i
    JOIN subsequent_admissions s ON i.patient_id = s.patient_id AND s.readmission_date > i.index_date
"""

print(con.execute(gold_query).df())